In [1]:
import chromadb
import re
import unicodedata
import requests
import psycopg
from sqlalchemy import create_engine, text
from striprtf.striprtf import rtf_to_text
import pandas as pd

# chroma_client = chromadb.HttpClient(host='localhost', port=8000)
# chroma_client.heartbeat()

# collections = chroma_client.list_collections()

In [ ]:

# engine = create_engine(DATABASE_URL)

# with engine.connect() as conn:
#     result = conn.execute(text("SELECT 1"))
#     print(result.scalar())

# docs = pd.read_sql(
#     "SELECT * FROM documents LIMIT 5",
#     engine
# )

# docs

1


In [2]:
from striprtf.striprtf import rtf_to_text
import glob

rtf_texts = []

for file_path in glob.glob("../chroma/data/*.rtf"):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        rtf_content = f.read()
        text = rtf_to_text(rtf_content)
        rtf_texts.append(text)

len(rtf_texts)

5

In [3]:
def clean_legal_text(text: str) -> str:
    """
    Clean court/legal text for RAG usage:
    - normalize unicode
    - remove non-breaking spaces
    - fix spaced words like
    - normalize whitespace
    - keep paragraph structure
    """

    if not text:
        return ""

    # 1. Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # 2. Replace non-breaking spaces and tabs
    text = text.replace("\xa0", " ").replace("\t", " ")

    # 3. Fix spaced-uppercase words (У Х В А Л А → УХВАЛА)
    text = re.sub(
        r'\b(?:[А-ЯІЇЄҐ]\s){2,}[А-ЯІЇЄҐ]\b',
        lambda m: m.group(0).replace(" ", ""),
        text
    )

    # 4. Remove excessive spaces
    text = re.sub(r'[ ]{2,}', ' ', text)

    # 5. Normalize newlines (max 2 in a row)
    text = re.sub(r'\n{3,}', '\n\n', text)

    # 6. Trim lines
    text = "\n".join(line.strip() for line in text.splitlines())

    # 7. Final trim
    return text.strip()

In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, 
                                          chunk_overlap=100,
                                          separators=["\n\n", "\n", ". ", " "])
chunks = splitter.split_text(clean_legal_text(rtf_texts[0]))

In [24]:
chunks

['Суддя Черняк В. Г.\n\nСправа No 646/449/25\nПровадження No 2/644/2708/25\n19.06.2025\n\nРІШЕННЯ\nІМЕНЕМУКРАЇНИ\n(заочне рішення)',
 '19 червня 2025 року м. Харків\nІндустріальний районний суд м. Харкова у складі:\nГоловуючого - судді Черняка В. Г.,\nза участю:\nсекретаря судових засідань –Юр`єва Є.Д.,\nрозглянувши у відкритому судовому засіданні в залі суду в м. Харкові в порядку спрощеного позовного провадження цивільну справу за позовом Товариства з обмеженою відповідальністю «Фінансова компанія «Кредит капітал» до ОСОБА_1 про стягнення заборгованості,-\nу с т а н о в и в :',
 'Позивач ТОВ "ФК «Кредит капітал» через систему «Електронний суд» звернулося до суду з позовом до ОСОБА_1 про стягнення заборгованості за кредитним договором. Позовні вимоги мотивовані тим, що 03.12.2022 року між Товариством з обмеженою відповідальністю «Лінеура Україна» та ОСОБА_1 укладено договір про надання коштів на умовах споживчого кредиту No 3307920',
 '. 13.09.2023 року ТОВ «Лінеура Україна» та ТОВ «Ф

In [ ]:
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

collection = chroma_client.create_collection(
    name="new_gen_docs",
    embedding_function=OpenAIEmbeddingFunction(
        api_key=
        model_name="text-embedding-3-large"
    )
)

In [17]:
collection.peek()

{'ids': [],
 'embeddings': array([], dtype=float64),
 'metadatas': [],
 'documents': [],
 'data': None,
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings']}

In [ ]:
# collection = chroma_client.get_collection(
#     name='test_collection_v2',
# )

In [25]:
docs.head(1)

,doc_id,court_code,judgment_code,justice_kind,category_code,cause_num,adjudication_date,receipt_date,judge,doc_url,status,date_publ
0,124212035,227,5.0,1.0,40356.0,149/4141/24,2025-01-02 00:00:00+02,2025-01-04 00:00:00+02,Олійник І. В.,http://od.reyestr.court.gov.ua/files/62/90134c...,1,2025-01-06 00:00:00+02


In [37]:
from tqdm import tqdm

doc_text = []
metadatas = []
ids = []

for _, row in tqdm(docs.iterrows()):

    cl_text = extract_rtf_text(row["doc_url"])
    doc_text.append(cl_text)

    metadatas.append({
        "doc_id": int(row["doc_id"]),
        "cause_num": row["cause_num"],
        "judge": row["judge"],
        "court_code": row["court_code"],
        "date": row["date_publ"]

    })

    ids.append(str(row["doc_id"]))

5it [00:00, 14.84it/s]


In [38]:
collection.add(
    documents=doc_text,
    metadatas=metadatas,
    ids=ids,
)

BadRequestError: Error code: 400 - {'error': {'message': "This model's maximum context length is 8192 tokens, however you requested 10290 tokens (10290 in your prompt; 0 for the completion). Please reduce your prompt; or completion length.", 'type': 'invalid_request_error', 'param': None, 'code': None}} in add.

In [5]:
collection.add(
    documents = [rtf_texts[0], rtf_texts[1], rtf_texts[2], rtf_texts[3]],
    metadatas = [{"source": "149/3813/25"},{"source": "149/4141/24"},{'source':'149/339/25'},{'source':'149/6/25'}],
    ids = ["id1", "id2", "id3", "id4"]
)

{'ids': ['id1', 'id2', 'id3', 'id4'],
 'embeddings': array([[ 0.02162955,  0.04352154, -0.05550636, ...,  0.00302354,
          0.0126956 ,  0.01563713],
        [ 0.02045577,  0.03636581, -0.05015524, ..., -0.01164706,
         -0.00625309,  0.02731786],
        [ 0.00720943,  0.01756578, -0.02543858, ..., -0.01258771,
          0.00152275,  0.03695174],
        [ 0.00384395,  0.05690486, -0.0745467 , ...,  0.01200441,
         -0.00283529,  0.01644803]], shape=(4, 1536)),
 'metadatas': [{'source': '149/3813/25'},
  {'source': '149/4141/24'},
  {'source': '149/339/25'},
  {'source': '149/6/25'}],
 'documents': ['\nУ Х В А Л А\nСправа № 149/3813/25\nПровадження №1-кс/149/687/25\n\n24.11.2025 р. \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 м. Хмільник\nХмільницький міськрайонний суд Вінницької області в складі:\nслідчого судді ОСОБА_1 ,\nза участі секретаря ОСОБА_2 ,\nпроку

In [59]:
q = """Вінницька область, м. Хмільник, вул. Столярчука, 4"""

In [60]:
collection.query(
    query_texts=[q],
    n_results=1
)

{'ids': [['id2']],
 'distances': [[0.5257132]],
 'embeddings': None,
 'metadatas': [[{'source': '149/4141/24'}]],
 'documents': [['\nСправа № 149/4141/24\nПровадження №2/149/196/25 \nНомер рядка звіту 48\n\xa0 У\xa0Х В\xa0А Л\xa0А \n"02" січня 2025 р. \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0м. Хмільник\nСуддя Хмільницького міськрайонного суду Вінницької області Олійник І. В., перевіривши матеріали справи за позовом Державної екологічної інспекції у Вінницькій області, третя особа, яка не заявляє самостійних вимог щодо предмета спору на стороні позивача: Уланівська сільська рада Хмільницького району Вінницької області до ОСОБА_1 про відшкодування шкоди, завданої державі внаслідок порушення природоохоронного законодавства,\n\nВСТАНОВИВ:\nДо Хмільницького міськрайонного суду Вінницької області надійшла вказана позовна заява.\nДослідивши матеріали, суддя дійшов висновку про передачу її за підсудністю, з огляду н

In [63]:
collection.get(
    where_document={"$contains": "Хмільник"},
    limit=2
)

{'ids': ['id1', 'id2'],
 'embeddings': None,
 'metadatas': [{'source': '149/3813/25'}, {'source': '149/4141/24'}],
 'documents': ['\nУ Х В А Л А\nСправа № 149/3813/25\nПровадження №1-кс/149/687/25\n\n24.11.2025 р. \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 м. Хмільник\nХмільницький міськрайонний суд Вінницької області в складі:\nслідчого судді ОСОБА_1 ,\nза участі секретаря ОСОБА_2 ,\nпрокурора ОСОБА_3 (приймає участь в режимі відеоконференції)\nслідчого ОСОБА_4 ,\nзахисника адвоката ОСОБА_5 ,\nпідозрюваного ОСОБА_6 ,\nрозглянувши у судовому засіданні клопотання слідчої СВ Хмільницького РВП ГУНП у Вінницькій області ОСОБА_7 про застосування запобіжного заходу у вигляді тримання під вартою, заявлене у кримінальному провадженні № 42023052210000216, внесеному до Єдиного реєстру досудових розслідувань 29.05.2023, за ознаками кримінального правопорушення, передбаченого ч. 4\x

4

In [8]:
print(clean_legal_text(rtf_texts[0]))

УХВАЛА
Справа No 149/3813/25
Провадження No1-кс/149/687/25

24.11.2025 р. м. Хмільник
Хмільницький міськрайонний суд Вінницької області в складі:
слідчого судді ОСОБА_1 ,
за участі секретаря ОСОБА_2 ,
прокурора ОСОБА_3 (приймає участь в режимі відеоконференції)
слідчого ОСОБА_4 ,
захисника адвоката ОСОБА_5 ,
підозрюваного ОСОБА_6 ,
розглянувши у судовому засіданні клопотання слідчої СВ Хмільницького РВП ГУНП у Вінницькій області ОСОБА_7 про застосування запобіжного заходу у вигляді тримання під вартою, заявлене у кримінальному провадженні No 42023052210000216, внесеному до Єдиного реєстру досудових розслідувань 29.05.2023, за ознаками кримінального правопорушення, передбаченого ч. 4 ст. 408 КК України, стосовно
ОСОБА_6 , ІНФОРМАЦІЯ_1 , громадянина України, не одруженого, проживаючого за адресою: АДРЕСА_1

ВСТАНОВИВ:
24.11.2025 до Хмільницького міськрайонного суду Вінницької області надійшло вказане клопотання, яке мотивовано тим, що ОСОБА_6 обґрунтовано підозрюється у вчиненні кримінал

In [7]:
print(rtf_texts[0])


У Х В А Л А
Справа № 149/3813/25
Провадження №1-кс/149/687/25

24.11.2025 р.                                                                 м. Хмільник
Хмільницький міськрайонний суд Вінницької області в складі:
слідчого судді ОСОБА_1 ,
за участі секретаря ОСОБА_2 ,
прокурора ОСОБА_3 (приймає участь в режимі відеоконференції)
слідчого ОСОБА_4 ,
захисника адвоката ОСОБА_5 ,
підозрюваного ОСОБА_6 ,
розглянувши у судовому засіданні клопотання слідчої СВ Хмільницького РВП ГУНП у Вінницькій області ОСОБА_7 про застосування запобіжного заходу у вигляді тримання під вартою, заявлене у кримінальному провадженні № 42023052210000216, внесеному до Єдиного реєстру досудових розслідувань 29.05.2023, за ознаками кримінального правопорушення, передбаченого ч. 4 ст. 408 КК України, стосовно
ОСОБА_6 , ІНФОРМАЦІЯ_1 , громадянина України, не одруженого, проживаючого за адресою: АДРЕСА_1 

ВСТАНОВИВ:
24.11.2025 до Хмільницького міськрайонного суду Вінницької області надійшло вказане клопотання, яке моти

{'ids': [['id3']],
 'distances': [[0.7251667]],
 'embeddings': None,
 'metadatas': [[{'source': '149/339/25'}]],
 'documents': [['\nСправа № 149/339/25\nПровадження №3/149/286/25 \nНомер рядка звіту 208\nП О С Т А Н О В А\nІМЕНЕМ УКРАЇНИ\n\xa0 \xa0 \xa0 \xa0\n20.02.2025 року \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0м. Хмільник\nСуддя Хмільницького міськрайонного суду Вінницької області Олійник І. В., розглянувши матеріали, що надійшли від ГУ ДПС у Вінницькій області про притягнення \nОСОБА_1 , ІНФОРМАЦІЯ_1 , громадянки України, РНОКПП НОМЕР_1 , заміжньої, яка проживає за адресою: АДРЕСА_1 \nдо адміністративної відповідальності за ч. 1 ст. 163-1 КУпАП, \n\nВ С Т А Н О В И В :\n13.01.2025, під час проведення документальної позапланової виїзної перевірки податкового, валютного та іншого законодавства, контроль за дотриманням якого покладено на контролюючі органи встановлен

In [29]:
collection.add(
    documents = [rtf_texts[0], rtf_texts[1], rtf_texts[2], rtf_texts[3]],
    metadatas = [{"source": "149/3813/25"},{"source": "149/4141/24"},{'source':'149/339/25'},{'source':'149/6/25'}],
    ids = ["id1", "id2", "id3", "id4"]
)

In [67]:
print(rtf_texts[0])


У Х В А Л А
Справа № 149/3813/25
Провадження №1-кс/149/687/25

24.11.2025 р.                                                                 м. Хмільник
Хмільницький міськрайонний суд Вінницької області в складі:
слідчого судді ОСОБА_1 ,
за участі секретаря ОСОБА_2 ,
прокурора ОСОБА_3 (приймає участь в режимі відеоконференції)
слідчого ОСОБА_4 ,
захисника адвоката ОСОБА_5 ,
підозрюваного ОСОБА_6 ,
розглянувши у судовому засіданні клопотання слідчої СВ Хмільницького РВП ГУНП у Вінницькій області ОСОБА_7 про застосування запобіжного заходу у вигляді тримання під вартою, заявлене у кримінальному провадженні № 42023052210000216, внесеному до Єдиного реєстру досудових розслідувань 29.05.2023, за ознаками кримінального правопорушення, передбаченого ч. 4 ст. 408 КК України, стосовно
ОСОБА_6 , ІНФОРМАЦІЯ_1 , громадянина України, не одруженого, проживаючого за адресою: АДРЕСА_1 

ВСТАНОВИВ:
24.11.2025 до Хмільницького міськрайонного суду Вінницької області надійшло вказане клопотання, яке моти

In [75]:
collection.similarity_search(query='№ 149/3813/25',k=2)

AttributeError: 'Collection' object has no attribute 'similarity_search'

In [73]:
collection.query(
  query_texts=["Справа № 149/3813/25"],
  n_results=1,
  where_document={"$contains": "search string"}
)

{'ids': [[]],
 'distances': [[]],
 'embeddings': None,
 'metadatas': [[]],
 'documents': [[]],
 'uris': None,
 'data': None,
 'included': ['metadatas', 'documents', 'distances']}

In [57]:
collection.query(
    query_embeddings=[[11.1, 12.1, 13.1]],
    n_results=5
)


InvalidArgumentError: Collection expecting embedding with dimension of 1536, got 3

In [49]:
collection.query(
    query_texts=["судова справа"],
    where={"city": "Хмільник"},
    n_results=10
)

{'ids': [[]],
 'distances': [[]],
 'embeddings': None,
 'metadatas': [[]],
 'documents': [[]],
 'uris': None,
 'data': None,
 'included': ['metadatas', 'documents', 'distances']}

In [44]:
from chromadb import Search, K, Knn, Rrf

In [46]:
# Dense semantic embeddings
dense_rank = Knn(
    query="machine learning research",  # Text query for dense embeddings
    key="#embedding",          # Default embedding field
    return_rank=True,
    limit=200                  # Consider top 200 candidates
)

# Sparse keyword embeddings
sparse_rank = Knn(
    query="machine learning research",  # Text query for sparse embeddings
    key="sparse_embedding",    # Metadata field for sparse vectors
    return_rank=True,
    limit=200
)

# Combine with RRF
hybrid_rank = Rrf(
    ranks=[dense_rank, sparse_rank],
    weights=[0.7, 0.3],       # 70% semantic, 30% keyword
    k=60
)

collection.search("справи із міста Хмільник")

AttributeError: 'str' object has no attribute 'to_dict'

In [2]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [3]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    host="localhost",
)

In [ ]:
collection.add(
    documents = [rtf_texts[0], rtf_texts[1], rtf_texts[2], rtf_texts[3]],
    metadatas = [{"source": "149/3813/25"},{"source": "149/4141/24"},{'source':'149/339/25'},{'source':'149/6/25'}],
    ids = ["id1", "id2", "id3", "id4"]
)

In [ ]:
dfsdf

'\nУ Х В А Л А\nСправа № 149/3813/25\nПровадження №1-кс/149/687/25\n\n24.11.2025 р. \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 м. Хмільник\nХмільницький міськрайонний суд Вінницької області в складі:\nслідчого судді ОСОБА_1 ,\nза участі секретаря ОСОБА_2 ,\nпрокурора ОСОБА_3 (приймає участь в режимі відеоконференції)\nслідчого ОСОБА_4 ,\nзахисника адвоката ОСОБА_5 ,\nпідозрюваного ОСОБА_6 ,\nрозглянувши у судовому засіданні клопотання слідчої СВ Хмільницького РВП ГУНП у Вінницькій області ОСОБА_7 про застосування запобіжного заходу у вигляді тримання під вартою, заявлене у кримінальному провадженні № 42023052210000216, внесеному до Єдиного реєстру досудових розслідувань 29.05.2023, за ознаками кримінального правопорушення, передбаченого ч. 4\xa0ст. 408 КК України, стосовно\nОСОБА_6 , ІНФОРМАЦІЯ_1 , громадянина України, не одруженого, проживаючого за адресою: АДРЕСА_1 \n\

In [17]:
from uuid import uuid4

from langchain_core.documents import Document

document_1 = Document(
    page_content=rtf_texts[0],
    metadata={"source": "149/3813/25"},
    id=1,
)

document_2 = Document(
    page_content=rtf_texts[1],
    metadata={"source": "149/4141/24"},
    id=2,
)

document_3 = Document(
    page_content=rtf_texts[2],
    metadata={"source": "149/339/25"},
    id=3,
)

document_4 = Document(
    page_content=rtf_texts[3],
    metadata={"source": "149/6/25"},
    id=4,
)


documents = [
    document_1,
    document_2,
    document_3,
    document_4
]
uuids = [str(uuid4()) for _ in range(len(documents))]

vector_store.add_documents(documents=documents, ids=uuids)

['2accfa83-4e00-45cc-8e1e-a7a7daf69c21',
 '5714d8ec-5d7d-44ab-bda8-65208be4b540',
 '01f35cea-aaff-4fd5-8332-c9f9111b129c',
 '7613c3c7-96b1-48dc-9d7a-c30c2492cec6']

In [21]:
vector_store.similarity_search(
    "149/3813/25",
    k=1
)

[Document(id='7613c3c7-96b1-48dc-9d7a-c30c2492cec6', metadata={'source': '149/6/25'}, page_content='\nСправа № 149/6/25\nПровадження №2/149/219/25 \nНомер рядка звіту 40 \nУ Х В А Л А\nпро відкриття провадження у справі\n14.01.2025 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 м. Хмільник\nСуддя Хмільницького міськрайонного суду Вінницької області Олійник І. В., отримавши позовну заяву ТОВ "ФК "Кредит-Капітал" до ОСОБА_1 про стягнення заборгованості,\nВСТАНОВИВ:\nДо Хмільницького міськрайонного суду Вінницької області надійшла вказана позовна заява,, яка підсудна цьому суду, відповідає вимогам ст.ст. 175-177 ЦПК України, підстави для залишення її без руху, повернення, відмови у відкритті провадження відсутні. \nЗа змістом ст.\xa0274\xa0ЦПК України\xa0дана справа підлягає розгляду в порядку спрощеного позовного провадження.\nВиходячи з\xa0викладеного,\xa0вважаю,\xa0що у\xa0справі слід\xa0відкрити провадження\xa0 тавідповідно до ч. 5 ст. 279 ЦПК України,

In [25]:
retriever = vector_store.as_retriever(
    search_type="mmr", search_kwargs={"k": 1, "fetch_k": 5}
)

retriever.invoke("№2/149/219/25")


[Document(id='01f35cea-aaff-4fd5-8332-c9f9111b129c', metadata={'source': '149/339/25'}, page_content='\nСправа № 149/339/25\nПровадження №3/149/286/25 \nНомер рядка звіту 208\nП О С Т А Н О В А\nІМЕНЕМ УКРАЇНИ\n\xa0 \xa0 \xa0 \xa0\n20.02.2025 року \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0 \xa0м. Хмільник\nСуддя Хмільницького міськрайонного суду Вінницької області Олійник І. В., розглянувши матеріали, що надійшли від ГУ ДПС у Вінницькій області про притягнення \nОСОБА_1 , ІНФОРМАЦІЯ_1 , громадянки України, РНОКПП НОМЕР_1 , заміжньої, яка проживає за адресою: АДРЕСА_1 \nдо адміністративної відповідальності за ч. 1 ст. 163-1 КУпАП, \n\nВ С Т А Н О В И В :\n13.01.2025, під час проведення документальної позапланової виїзної перевірки податкового, валютного та іншого законодавства, контроль за дотриманням якого покладено на контролюючі органи встановлено порушення Хмільницькою між

#### Neon Connect

1


,doc_id,court_code,judgment_code,justice_kind,category_code,cause_num,adjudication_date,receipt_date,judge,doc_url,status,date_publ
0,124212035,227,5.0,1.0,40356.0,149/4141/24,2025-01-02 00:00:00+02,2025-01-04 00:00:00+02,Олійник І. В.,http://od.reyestr.court.gov.ua/files/62/90134c...,1,2025-01-06 00:00:00+02
1,129579772,814,5.0,2.0,40476.0,334/6318/25,2025-08-14 00:00:00+03,2025-08-18 00:00:00+03,Турбіна Т. Ф.,http://od.reyestr.court.gov.ua/files/64/20b931...,1,2025-08-19 00:00:00+03
2,129526117,1521,5.0,1.0,40352.0,509/4270/25,2025-08-14 00:00:00+03,2025-08-14 00:00:00+03,Кириченко П. Л.,http://od.reyestr.court.gov.ua/files/64/5981b7...,1,2025-08-15 00:00:00+03
3,128243120,2029,3.0,1.0,40348.0,646/449/25,2025-06-19 00:00:00+03,2025-06-19 00:00:00+03,Черняк В. Г.,http://od.reyestr.court.gov.ua/files/64/270b13...,1,2025-06-20 00:00:00+03
4,126288303,5026,5.0,3.0,40222.0,925/1660/24,2025-04-02 00:00:00+03,2025-04-02 00:00:00+03,Гладун А. І.,http://od.reyestr.court.gov.ua/files/63/5821cb...,1,2025-04-03 00:00:00+03
